# Building Change Detection with Early Fusion & Siamese  Architectures
## Google Colab Training Notebook (Hugging Face Dataset)

This notebook trains the change detection model using the OSCD dataset from Hugging Face.

**Before starting:**
1. Runtime → Change runtime type → Hardware accelerator → **GPU** → Save
2. Run cells in order from top to bottom

## Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install datasets huggingface_hub tqdm scikit-learn matplotlib -q

import torch
print(f" Dependencies installed!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 2: Set Hugging Face Token

To add a token:
1. Click the 🔑 icon on the left sidebar
2. Add new secret: Name = `HF_TOKEN`, Value = your token from https://huggingface.co/settings/tokens
3. Toggle "Notebook access" ON

###**This is optional** - the dataset will download without it, but a token makes it faster.

In [ ]:
from google.colab import files

print("Upload your Python files --not siem or TwoCh--  ...")
uploaded = files.upload()

# List uploaded files
!ls -lh

In [ ]:
# Try to get token from Colab secrets (optional)
from google.colab import userdata
import os

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print("Hugging Face token loaded from secrets!")
except:
    print("⚠️ No token found (that's okay, dataset will still download)")
    print("   To add: Click 🔑 icon → Add secret → Name: HF_TOKEN")

## Step 3: Download OSCD Dataset from Hugging Face

In [ ]:
from datasets import load_dataset

print("Downloading OSCD dataset from Hugging Face...")
print("This may take 2-3 minutes...\n")

dataset = load_dataset("blanchon/OSCD_RGB")

print(f"\n✓ Dataset downloaded!")
print(f"Train cities: {len(dataset['train'])}")
print(f"Test cities: {len(dataset['test'])}")
print(f"\nDataset structure: {dataset['train'].features}")

## Step 4: Create Dataset Loader for Hugging Face Format

In [ ]:
import torch
from torch.utils.data import Dataset
import numpy as np
from PIL import Image

class OSCDDatasetHF(Dataset):
    """
    OSCD Dataset loader for Hugging Face datasets format
    Works with the blanchon/OSCD_RGB dataset
    """

    def __init__(self, hf_dataset, patch_size=15, stride=5, use_augmentation=True):
        self.hf_dataset = hf_dataset
        self.patch_size = patch_size
        self.stride = stride
        self.use_augmentation = use_augmentation

        self.patches = []
        self._extract_patches()

        print(f"Loaded {len(self.patches)} patches")

    def _extract_patches(self):
        """Extract patches from all images in the dataset"""

        for idx in range(len(self.hf_dataset)):
            sample = self.hf_dataset[idx]

            # Get images and label from the HF dataset
            # Original: images = sample['images'] # List of 2 PIL Images
            images = [sample['image1'], sample['image2']] # Corrected to use image1 and image2
            label_img = sample['mask']  # Corrected to use 'mask' for the label image
            city = sample.get('pair_id', f'city_{idx}')  # City name

            # Convert PIL images to numpy arrays
            img1 = np.array(images[0])  # First image (T1)
            img2 = np.array(images[1])  # Second image (T2)
            label_map = np.array(label_img)

            # Convert to float and normalize to [0, 1]
            img1 = img1.astype(np.float32) / 255.0
            img2 = img2.astype(np.float32) / 255.0

            # Convert from (H, W, C) to (C, H, W) for PyTorch
            if len(img1.shape) == 3:  # RGB
                img1 = np.transpose(img1, (2, 0, 1))
                img2 = np.transpose(img2, (2, 0, 1))
                C, H, W = img1.shape
            else:  # Grayscale
                H, W = img1.shape
                C = 1
                img1 = img1.reshape(1, H, W)
                img2 = img2.reshape(1, H, W)

            # Convert label to binary (0: no change, 1: change)
            # White (255) = change, black (0) = no change
            labels = (label_map > 0).astype(np.int64) # Changed from > 128 to > 0

            # Extract patches
            for y in range(0, H - self.patch_size + 1, self.stride):
                for x in range(0, W - self.patch_size + 1, self.stride):
                    # Extract patch
                    patch1 = img1[:, y:y+self.patch_size, x:x+self.patch_size]
                    patch2 = img2[:, y:y+self.patch_size, x:x+self.patch_size]

                    # Get label for center pixel
                    center_y = y + self.patch_size // 2
                    center_x = x + self.patch_size // 2
                    label = labels[center_y, center_x]

                    self.patches.append({
                        'patch1': patch1,
                        'patch2': patch2,
                        'label': label,
                        'city': city
                    })

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        patch_data = self.patches[idx]

        patch1 = torch.from_numpy(patch_data['patch1'].copy()).float()
        patch2 = torch.from_numpy(patch_data['patch2'].copy()).float()
        label = torch.tensor(patch_data['label'], dtype=torch.long)

        # Data augmentation (only if enabled)
        if self.use_augmentation:
            # Random horizontal flip
            if np.random.rand() > 0.5:
                patch1 = torch.flip(patch1, [-1])
                patch2 = torch.flip(patch2, [-1])

            # Random vertical flip
            if np.random.rand() > 0.5:
                patch1 = torch.flip(patch1, [-2])
                patch2 = torch.flip(patch2, [-2])

            # Random rotation (0, 90, 180, 270 degrees)
            k = np.random.randint(0, 4)
            if k > 0:
                patch1 = torch.rot90(patch1, k, [-2, -1])
                patch2 = torch.rot90(patch2, k, [-2, -1])

        return patch1, patch2, label


def get_class_weights(dataset):
    """Calculate class weights for handling imbalanced dataset"""
    labels = [patch['label'] for patch in dataset.patches]
    unique, counts = np.unique(labels, return_counts=True)

    # Inverse frequency weighting
    total = len(labels)
    # If only one class is present, we cannot calculate inverse frequency weights for all classes
    # In such a case, set weights to [1.0] to indicate no special weighting, or handle gracefully
    if len(unique) < 2:
        weights = np.ones(2) # Assign weight 1.0 to both classes if one is missing
        print("Warning: Only one class found in dataset. Assigning equal weights for CrossEntropyLoss.")
    else:
        weights = total / (len(unique) * counts)

    print(f"\nClass distribution:")
    for cls, count in zip(unique, counts):
        percentage = 100 * count / total
        cls_name = "No Change" if cls == 0 else "Change"
        print(f"  {cls_name} (class {cls}): {count:,} patches ({percentage:.2f}%)")

    print(f"\nClass weights: {dict(zip(unique, weights))}")

    return torch.FloatTensor(weights)


print("✓ Dataset loader created!")

## Step 5: Load and Process the Dataset

In [ ]:
print("Loading and processing datasets...")
print("This may take 5-10 minutes...\n")

# Create train dataset (small stride for more data)
train_dataset = OSCDDatasetHF(
    hf_dataset=dataset['train'],
    patch_size=15,
    stride=5,  # Small stride = more patches = better training
    use_augmentation=True
)

# Create test dataset (larger stride for faster validation)
test_dataset = OSCDDatasetHF(
    hf_dataset=dataset['test'],
    patch_size=15,
    stride=15,  # Larger stride = fewer patches = faster validation
    use_augmentation=False
)

print(f"\n✓ Datasets created successfully!")
print(f"Training patches: {len(train_dataset):,}")
print(f"Test patches: {len(test_dataset):,}")

# Calculate class weights for imbalanced data
class_weights = get_class_weights(train_dataset)

## Step 6: Visualize Sample Patches

In [ ]:
import matplotlib.pyplot as plt

print("Visualizing random samples...\n")

fig, axes = plt.subplots(5, 3, figsize=(12, 15))

for i in range(5):
    # Get random sample
    idx = np.random.randint(len(train_dataset))
    patch1, patch2, label = train_dataset[idx]

    # Convert to numpy for plotting
    if patch1.shape[0] == 3:  # RGB
        img1 = patch1.permute(1, 2, 0).numpy()
        img2 = patch2.permute(1, 2, 0).numpy()
    else:  # Grayscale
        img1 = patch1.squeeze().numpy()
        img2 = patch2.squeeze().numpy()

    # Clip to [0, 1] for display
    img1 = np.clip(img1, 0, 1)
    img2 = np.clip(img2, 0, 1)

    # Plot
    if patch1.shape[0] == 3:
        axes[i, 0].imshow(img1)
        axes[i, 1].imshow(img2)
    else:
        axes[i, 0].imshow(img1, cmap='gray')
        axes[i, 1].imshow(img2, cmap='gray')

    axes[i, 0].set_title(f'Sample {i+1}: Image T1')
    axes[i, 1].set_title('Image T2')
    axes[i, 0].axis('off')
    axes[i, 1].axis('off')

    # Label
    label_text = "CHANGE" if label == 1 else "NO CHANGE"
    color = 'red' if label == 1 else 'green'
    axes[i, 2].text(0.5, 0.5, label_text,
                   ha='center', va='center',
                   fontsize=16, color=color, weight='bold')
    axes[i, 2].set_title('Label')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig('dataset_samples.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Visualization complete!")

## Step 7: Upload Model File

Upload your **TwoChNet_15.py** file here

## Step 7/2: Upload Model File

Upload your **SiamNet_15.py** file here

In [ ]:
from google.colab import files

print("upload SiamNet_15.py")
uploaded = files.upload()

print("\n✓ Model file uploaded!")

TypeError: 'NoneType' object is not subscriptable

## Step 8: Verify Model Loads Correctly

In [ ]:
from SiamNet_15 import SiamNet_15

# Create model
model = SiamNet_15(n_in=3)  # 3 channels per image = 6 total

# Set model to evaluation mode for testing the forward pass
model.eval()

# Test forward pass
test_patch1, test_patch2, test_label = train_dataset[0]
test_patch1 = test_patch1.unsqueeze(0)  # Add batch dimension
test_patch2 = test_patch2.unsqueeze(0)

output = model(test_patch1, test_patch2)

print(f"✓ Model loaded and tested successfully!")
print(f"Input shape (patch 1): {test_patch1.shape}")
print(f"Input shape (patch 2): {test_patch2.shape}")
print(f"Output shape: {output.shape}")
print(f"Output (logits): {output}")
print(f"True label: {test_label}")

## Step 9: Setup Training

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Create model
model = SiamNet_15(n_in=3)
model = model.to(device)

# Print model info
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Loss and optimizer
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)

print("\n" + "="*60)
print("READY TO START TRAINING")
print("="*60)
print(f"Training patches: {len(train_dataset):,}")
print(f"Test patches: {len(test_dataset):,}")
print(f"Batches per epoch (train): {len(train_loader)}")
print(f"Batches per epoch (test): {len(test_loader)}")
print(f"Batch size: 128")
print(f"Learning rate: 0.001")
print(f"Device: {device}")
print("="*60 + "\n")

print("Run the next cell to start training!")

## Step 10: Training Loop

**This will take approximately 2-3 hours on Colab's free GPU**

Target: ~83.63% validation accuracy (from paper)

In [ ]:
num_epochs = 50
best_val_acc = 0
train_losses = []
val_losses = []
train_accs = []
val_accs = []

print("Starting training...\n")

for epoch in range(1, num_epochs + 1):
    # =================== TRAINING ===================
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{num_epochs} [Train]')
    for batch_idx, (patch1, patch2, labels) in enumerate(pbar):
        # Move to device
        patch1 = patch1.to(device)
        patch2 = patch2.to(device)
        labels = labels.to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(patch1, patch2)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # Update progress bar
        pbar.set_postfix({
            'loss': f'{running_loss/(batch_idx+1):.4f}',
            'acc': f'{100.*correct/total:.2f}%'
        })

    train_loss = running_loss / len(train_loader)
    train_acc = 100. * correct / total
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # =================== VALIDATION ===================
    model.eval()
    running_loss = 0
    correct = 0
    total = 0
    class_correct = [0, 0]
    class_total = [0, 0]

    with torch.no_grad():
        pbar = tqdm(test_loader, desc=f'Epoch {epoch}/{num_epochs} [Val]')
        for batch_idx, (patch1, patch2, labels) in enumerate(pbar):
            # Move to device
            patch1 = patch1.to(device)
            patch2 = patch2.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(patch1, patch2)
            loss = criterion(outputs, labels)

            # Statistics
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            # Per-class accuracy
            for i in range(2):
                mask = (labels == i)
                class_correct[i] += (predicted[mask] == i).sum().item()
                class_total[i] += mask.sum().item()

            # Update progress bar
            pbar.set_postfix({
                'loss': f'{running_loss/(batch_idx+1):.4f}',
                'acc': f'{100.*correct/total:.2f}%'
            })

    val_loss = running_loss / len(test_loader)
    val_acc = 100. * correct / total
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    # Per-class accuracies
    no_change_acc = 100. * class_correct[0] / max(class_total[0], 1)
    change_acc = 100. * class_correct[1] / max(class_total[1], 1)

    # Print epoch summary
    print(f'\nEpoch {epoch}/{num_epochs} Summary:')
    print(f'  Train - Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%')
    print(f'  Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%')
    print(f'  Val   - No-Change: {no_change_acc:.2f}%, Change: {change_acc:.2f}%')

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_acc': val_acc,
            'no_change_acc': no_change_acc,
            'change_acc': change_acc,
        }, 'best_model.pth')
        print(f'  ✓✓✓ Saved best model (Val Acc: {val_acc:.2f}%) ✓✓✓')

    # Plot training progress every 10 epochs
    if epoch % 10 == 0:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        # Loss plot
        ax1.plot(range(1, epoch+1), train_losses, label='Train Loss', marker='o', linewidth=2)
        ax1.plot(range(1, epoch+1), val_losses, label='Val Loss', marker='s', linewidth=2)
        ax1.set_xlabel('Epoch', fontsize=12)
        ax1.set_ylabel('Loss', fontsize=12)
        ax1.set_title('Training Progress - Loss', fontsize=14, weight='bold')
        ax1.legend(fontsize=11)
        ax1.grid(True, alpha=0.3)

        # Accuracy plot
        ax2.plot(range(1, epoch+1), train_accs, label='Train Acc', marker='o', linewidth=2)
        ax2.plot(range(1, epoch+1), val_accs, label='Val Acc', marker='s', linewidth=2)
        ax2.axhline(y=83.63, color='r', linestyle='--', linewidth=2, label='Paper Baseline (83.63%)')
        ax2.set_xlabel('Epoch', fontsize=12)
        ax2.set_ylabel('Accuracy (%)', fontsize=12)
        ax2.set_title('Training Progress - Accuracy', fontsize=14, weight='bold')
        ax2.legend(fontsize=11)
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(f'training_progress_epoch_{epoch}.png', dpi=150, bbox_inches='tight')
        plt.show()

    print('-' * 60)

# Training complete!
print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")
print(f"Target from paper: 83.63%")
print(f"Difference: {best_val_acc - 83.63:+.2f}%")

if best_val_acc >= 81.0:
    print("\n🎉 SUCCESS! Your model achieved good performance!")
elif best_val_acc >= 75.0:
    print("\n✓ Good start! Consider training for more epochs or adjusting hyperparameters.")
else:
    print("\n⚠️ Accuracy is lower than expected. Check the training curves for issues.")

print("="*60)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from SiamNet_15 import SiamNet_15

# 1. Setup Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Initialize Model
model = SiamNet_15(n_in=3).to(device)

# 3. Setup Loss and Optimizer (using the corrected class weights)
# class_weights should be [0.5118, 21.6478] based on your previous output
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)

print(f"✓ Model initialized on {device}")
print("You can now run the Step 10: Training Loop cell.")

In [ ]:
print("Re-calculating class weights and resetting training parameters...")

# 1. Re-calculate class weights using the corrected dataset
class_weights = get_class_weights(train_dataset)

# 2. Re-initialize the model to be safe
model = SiamNet_15(n_in=3).to(device)

# 3. Re-initialize the criterion with the CORRECT weight shape [2]
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

# 4. Re-initialize the optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)

print("\n✓ Success! Weights updated to shape:", class_weights.shape)
print("You can now run the Training Loop cell again.")

## Step 11: Final Results and Evaluation

In [ ]:
# Plot final training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss
ax1.plot(range(1, num_epochs+1), train_losses, label='Train Loss', marker='o', linewidth=2)
ax1.plot(range(1, num_epochs+1), val_losses, label='Val Loss', marker='s', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Final Training Curves - Loss', fontsize=14, weight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(range(1, num_epochs+1), train_accs, label='Train Acc', marker='o', linewidth=2)
ax2.plot(range(1, num_epochs+1), val_accs, label='Val Acc', marker='s', linewidth=2)
ax2.axhline(y=83.63, color='r', linestyle='--', linewidth=2, label='Paper Baseline (83.63%)')
ax2.axhline(y=best_val_acc, color='g', linestyle=':', linewidth=2, label=f'Best Val Acc ({best_val_acc:.2f}%)')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Final Training Curves - Accuracy', fontsize=14, weight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('final_training_curves.png', dpi=200, bbox_inches='tight')
plt.show()

print("✓ Final plots saved!")

## Step 12: Test Some Predictions

In [ ]:
# Load best model
checkpoint = torch.load('best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("Testing model on random samples...\n")

fig, axes = plt.subplots(5, 4, figsize=(14, 16))

correct_predictions = 0

for i in range(5):
    # Get random test sample
    idx = np.random.randint(len(test_dataset))
    patch1, patch2, label = test_dataset[idx]

    # Predict
    with torch.no_grad():
        p1 = patch1.unsqueeze(0).to(device)
        p2 = patch2.unsqueeze(0).to(device)
        output = model(p1, p2)
        prob = torch.softmax(output, dim=1)
        pred = output.argmax(1).item()
        confidence = prob[0, pred].item()

    # Track accuracy
    if pred == label:
        correct_predictions += 1

    # Plot
    if patch1.shape[0] == 3:
        img1 = patch1.permute(1, 2, 0).numpy()
        img2 = patch2.permute(1, 2, 0).numpy()
    else:
        img1 = patch1.squeeze().numpy()
        img2 = patch2.squeeze().numpy()

    img1 = np.clip(img1, 0, 1)
    img2 = np.clip(img2, 0, 1)

    if patch1.shape[0] == 3:
        axes[i, 0].imshow(img1)
        axes[i, 1].imshow(img2)
    else:
        axes[i, 0].imshow(img1, cmap='gray')
        axes[i, 1].imshow(img2, cmap='gray')

    axes[i, 0].set_title(f'Sample {i+1}: Image T1', fontsize=11)
    axes[i, 1].set_title('Image T2', fontsize=11)
    axes[i, 0].axis('off')
    axes[i, 1].axis('off')

    # Ground truth
    gt_text = "CHANGE" if label == 1 else "NO CHANGE"
    axes[i, 2].text(0.5, 0.5, f'Ground Truth:\n{gt_text}',
                   ha='center', va='center', fontsize=12, weight='bold')
    axes[i, 2].set_title('Ground Truth', fontsize=11)
    axes[i, 2].axis('off')

    # Prediction
    pred_text = "CHANGE" if pred == 1 else "NO CHANGE"
    color = 'green' if pred == label else 'red'
    status = '✓ CORRECT' if pred == label else '✗ WRONG'
    axes[i, 3].text(0.5, 0.5, f'Prediction:\n{pred_text}\n\n{status}\n({confidence*100:.1f}%)',
                   ha='center', va='center', fontsize=12, color=color, weight='bold')
    axes[i, 3].set_title('Prediction', fontsize=11)
    axes[i, 3].axis('off')

plt.suptitle(f'Model Predictions (Accuracy on these samples: {correct_predictions}/5)',
             fontsize=16, weight='bold', y=0.995)
plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Sample predictions: {correct_predictions}/5 correct")

## Step 13: Download Results

In [ ]:
from google.colab import files
import os

print("Preparing files for download...\n")

# Download best model
if os.path.exists('best_model.pth'):
    files.download('best_model.pth')
    print("✓ Downloaded: best_model.pth")

# Download final training plot
if os.path.exists('final_training_curves.png'):
    files.download('final_training_curves.png')
    print("✓ Downloaded: final_training_curves.png")

# Download sample predictions
if os.path.exists('sample_predictions.png'):
    files.download('sample_predictions.png')
    print("✓ Downloaded: sample_predictions.png")

# Download dataset samples
if os.path.exists('dataset_samples.png'):
    files.download('dataset_samples.png')
    print("✓ Downloaded: dataset_samples.png")

# Create summary text file
with open('training_summary.txt', 'w') as f:
    f.write("="*60 + "\n")
    f.write("BUILDING CHANGE DETECTION - TRAINING SUMMARY\n")
    f.write("="*60 + "\n\n")
    f.write("Model: Early Fusion (TwoChNet_15)\n")
    f.write("Dataset: OSCD (Onera Satellite Change Detection)\n")
    f.write(f"Training patches: {len(train_dataset):,}\n")
    f.write(f"Test patches: {len(test_dataset):,}\n")
    f.write(f"\nTraining Configuration:\n")
    f.write(f"  Epochs: {num_epochs}\n")
    f.write(f"  Batch size: 128\n")
    f.write(f"  Learning rate: 0.001\n")
    f.write(f"  Optimizer: Adam\n")
    f.write(f"  Loss: Weighted Cross-Entropy\n")
    f.write(f"\nFinal Results:\n")
    f.write(f"  Best Validation Accuracy: {best_val_acc:.2f}%\n")
    f.write(f"  Paper Baseline: 83.63%\n")
    f.write(f"  Difference: {best_val_acc - 83.63:+.2f}%\n")
    f.write(f"\nPer-Class Performance (Best Epoch):\n")
    f.write(f"  No-Change Accuracy: {checkpoint['no_change_acc']:.2f}%\n")
    f.write(f"  Change Accuracy: {checkpoint['change_acc']:.2f}%\n")
    f.write(f"\nTarget from Paper:\n")
    f.write(f"  Overall: 83.63%\n")
    f.write(f"  No-Change: 83.71%\n")
    f.write(f"  Change: 82.14%\n")
    f.write("\n" + "="*60 + "\n")

files.download('training_summary.txt')
print("✓ Downloaded: training_summary.txt")

print("\n" + "="*60)
print("ALL FILES DOWNLOADED!")
print("="*60)
print("\nYou now have:")
print("  1. Trained model (best_model.pth)")
print("  2. Training curves (final_training_curves.png)")
print("  3. Sample predictions (sample_predictions.png)")
print("  4. Dataset samples (dataset_samples.png)")
print("  5. Training summary (training_summary.txt)")
print("\nUse these for your project report!")
print("="*60)